In [ ]:
# Install packages with compatible versions for Colab
!pip install --upgrade pip
!pip install datasets transformers sacrebleu sentencepiece
# Install PyTorch version compatible with Colab's CUDA
!pip install --force-reinstall torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0

# Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import os
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.nn.utils.rnn import pad_sequence

## Check CUDA

In [ ]:
# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Settings

In [ ]:
# Define constants
BATCH_SIZE = 32
MAX_SEQ_LEN = 128  # Reasonable length for translation
EMBEDDING_SIZE = 512
NUM_HEADS = 8
FFN_HID_DIM = 2048
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
NUM_EPOCHS = 10
LEARNING_RATE = 0.0001
SUBSET_SIZE = 100000  # Use a subset of the data to fit on 3070

In [ ]:
# Create directories for saving data and models
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Data

## Download

In [ ]:
# Download and preprocess WMT14 English-French dataset using Hugging Face datasets
print("Loading WMT14 English-French dataset...")
dataset = load_dataset("wmt14", "fr-en")
print("Dataset loaded successfully!")

# Print dataset info
print(f"Dataset structure: {dataset}")

## Tokenize

In [ ]:
# Create tokenizers using Hugging Face transformers
tokenizer_en = AutoTokenizer.from_pretrained("t5-small")  # Using t5 tokenizer for English
tokenizer_fr = AutoTokenizer.from_pretrained("t5-small")  # Using t5 tokenizer for French

In [ ]:
# Function to tokenize the datasets
def tokenize_dataset(examples):
    en_texts = []
    fr_texts = []
    
    # Extract texts from the correct structure
    for item in examples["translation"]:
        en_texts.append(item["en"])
        fr_texts.append(item["fr"])
    
    en_tokenized = tokenizer_en(en_texts, truncation=True, max_length=MAX_SEQ_LEN-2)
    fr_tokenized = tokenizer_fr(fr_texts, truncation=True, max_length=MAX_SEQ_LEN-2)
    
    return {
        "en_tokens": en_tokenized["input_ids"],
        "fr_tokens": fr_tokenized["input_ids"]
    }

## Prorcess

In [ ]:
# # Tokenize train, validation and test datasets
# tokenized_datasets = dataset.map(tokenize_dataset, batched=True)

In [ ]:
# Tokenize datasets - ONLY PROCESS WHAT WE NEED
# We'll limit the processing to just what we need for testing
print("Tokenizing a limited subset of the dataset...")
tokenized_datasets = {}

# Process only a portion of each split
for split in dataset.keys():
    # Take only the first N examples from each split
    limit = SUBSET_SIZE if split == "train" else (SUBSET_SIZE // 10 if split == "validation" else SUBSET_SIZE // 20)
    small_dataset = dataset[split].select(range(min(limit * 2, len(dataset[split]))))  # Double the size to account for filtering
    tokenized_datasets[split] = small_dataset.map(tokenize_dataset, batched=True)
    print(f"Processed {len(tokenized_datasets[split])} samples for {split}")

## Filter by length and limit data size

In [ ]:
# Create a function to filter by length and limit dataset size
def prepare_dataset(dataset_split, size):
    data = tokenized_datasets[dataset_split].select(range(min(size, len(tokenized_datasets[dataset_split]))))
    print(f"{dataset_split} samples: {len(data)}")
    
    src_data = []
    tgt_data = []
    
    for item in data:
        en_tokens = item["en_tokens"]
        fr_tokens = item["fr_tokens"]
        
        # Skip samples that are too long
        if len(en_tokens) <= MAX_SEQ_LEN and len(fr_tokens) <= MAX_SEQ_LEN:
            src_data.append(torch.tensor(en_tokens, dtype=torch.long))
            tgt_data.append(torch.tensor(fr_tokens, dtype=torch.long))
    
    return src_data, tgt_data

## Prepare data

In [ ]:
# Prepare the datasets
train_src, train_tgt = prepare_dataset("train", SUBSET_SIZE)
val_src, val_tgt = prepare_dataset("validation", SUBSET_SIZE // 10)
test_src, test_tgt = prepare_dataset("test", SUBSET_SIZE // 20)

# Get vocabulary sizes from tokenizers
src_vocab_size = tokenizer_en.vocab_size
tgt_vocab_size = tokenizer_fr.vocab_size
print(f"English vocabulary size: {src_vocab_size}")
print(f"French vocabulary size: {tgt_vocab_size}")

# Special token indices
PAD_IDX = tokenizer_en.pad_token_id
BOS_IDX = tokenizer_en.bos_token_id if tokenizer_en.bos_token_id else tokenizer_en.cls_token_id
EOS_IDX = tokenizer_en.eos_token_id if tokenizer_en.eos_token_id else tokenizer_en.sep_token_id

In [ ]:
# Create batch function with padding
def collate_batch(batch):
    src_batch, tgt_batch = [], []
    for src, tgt in batch:
        src_batch.append(src)
        tgt_batch.append(tgt)
    
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX, batch_first=True)
    
    return src_batch, tgt_batch

In [ ]:
# Create a custom dataset for handling tensor lists
class TranslationDataset(data.Dataset):
    def __init__(self, src_tensors, tgt_tensors):
        assert len(src_tensors) == len(tgt_tensors), "Source and target lists must have same length"
        self.src_tensors = src_tensors
        self.tgt_tensors = tgt_tensors
    
    def __len__(self):
        return len(self.src_tensors)
    
    def __getitem__(self, idx):
        return self.src_tensors[idx], self.tgt_tensors[idx]

In [ ]:
# Create DataLoaders with custom dataset
train_dataset = TranslationDataset(train_src, train_tgt)
train_loader = data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)

val_dataset = TranslationDataset(val_src, val_tgt)
val_loader = data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

test_dataset = TranslationDataset(test_src, test_tgt)
test_loader = data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

# Transformer

## Using Library

In [ ]:
# Transformer Model Definition
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, nhead, num_encoder_layers,
                 num_decoder_layers, dim_feedforward, max_seq_length, dropout=0.1):
        super(TransformerModel, self).__init__()
        
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        
        self.out = nn.Linear(d_model, tgt_vocab_size)
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None,
                src_padding_mask=None, tgt_padding_mask=None,
                memory_key_padding_mask=None):
        
        src_emb = self.positional_encoding(self.src_embedding(src))
        tgt_emb = self.positional_encoding(self.tgt_embedding(tgt))
        
        output = self.transformer(src_emb, tgt_emb, src_mask, tgt_mask,
                                 None, src_padding_mask, tgt_padding_mask, memory_key_padding_mask)
        
        return self.out(output)
    
    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask
    
    def create_padding_mask(self, seq, pad_idx):
        return (seq == pad_idx)

## Implementation from Scrath

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

### ScaledDot

In [ ]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super(ScaledDotProductAttention, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        # query, key, value: [batch_size, num_heads, seq_len, d_k]
        d_k = query.size(-1)
        
        # Calculate attention scores
        # [batch_size, num_heads, seq_len_q, seq_len_k]
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
        
        # Apply mask (if provided)
        if mask is not None:
            # Handle different mask types based on dimensions
            if mask.dim() == 2 and mask.size(0) == mask.size(1):  # Causal mask [tgt_len, tgt_len]
                # Expand causal mask to match scores dimensions
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, tgt_len, tgt_len]
                scores = scores.masked_fill(mask, -1e9)
            elif mask.dim() == 2:  # Padding mask [batch_size, seq_len]
                # Convert padding mask for proper broadcasting
                mask = mask.unsqueeze(1).unsqueeze(2)  # [batch_size, 1, 1, seq_len]
                scores = scores.masked_fill(mask, -1e9)
            elif mask.dim() > 2:  # Already expanded mask
                scores = scores.masked_fill(mask, -1e9)
        
        # Apply softmax and dropout
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Calculate weighted sum
        output = torch.matmul(attn_weights, value)
        
        return output, attn_weights

### MultiHeadAttention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.attention = ScaledDotProductAttention(dropout)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model)
        
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        residual = query
        
        # Linear projections and reshape
        q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Apply attention
        output, attn_weights = self.attention(q, k, v, mask)
        
        # Reshape and final linear projection
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(output)
        output = self.dropout(output)
        
        # Add residual and normalize
        output = self.layer_norm(residual + output)
        
        return output, attn_weights

### Positionwise Feed Forward

In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model)
        
    def forward(self, x):
        residual = x
        output = self.w_2(F.relu(self.w_1(x)))
        output = self.dropout(output)
        output = self.layer_norm(residual + output)
        return output

### Encoder layer

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        
    def forward(self, x, mask=None):
        # Self-attention block
        enc_output, _ = self.self_attn(x, x, x, mask)
        
        # Feed-forward block
        enc_output = self.feed_forward(enc_output)
        
        return enc_output

### Decoder layer

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        
    def forward(self, x, enc_output, self_attn_mask=None, cross_attn_mask=None):
        # Combine tgt_causal_mask and tgt_key_padding_mask if needed
        # For self-attention masking
        
        # Self-attention block
        dec_output, _ = self.self_attn(x, x, x, self_attn_mask)
        
        # Cross-attention block
        dec_output, _ = self.cross_attn(dec_output, enc_output, enc_output, cross_attn_mask)
        
        # Feed-forward block
        dec_output = self.feed_forward(dec_output)
        
        return dec_output

### Encoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_seq_len, dropout=0.1):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)
        self.dropout = nn.Dropout(dropout)
        
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
    def forward(self, x, mask=None):
        # Convert word indices to embeddings [batch_size, seq_len, d_model]
        x = self.embedding(x)
        # Add positional encoding
        x = self.positional_encoding(x)
        # Apply dropout
        x = self.dropout(x)
        
        # Apply encoder layers
        for layer in self.layers:
            x = layer(x, mask)
            
        return x

### Decoder

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_seq_len, dropout=0.1):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)
        self.dropout = nn.Dropout(dropout)
        
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
    def forward(self, x, enc_output, tgt_mask=None, cross_attn_mask=None):
        # Convert word indices to embeddings [batch_size, seq_len, d_model]
        x = self.embedding(x)
        # Add positional encoding
        x = self.positional_encoding(x)
        # Apply dropout
        x = self.dropout(x)
        
        # Apply decoder layers
        for layer in self.layers:
            x = layer(x, enc_output, tgt_mask, cross_attn_mask)
            
        return x

### Position Encoder

In [ ]:
# Transformer Model Definition
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

### Transformer

In [ ]:
class CustomTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, 
                 num_encoder_layers, num_decoder_layers, d_ff, max_seq_length, dropout=0.1):
        super(CustomTransformer, self).__init__()
        
        self.encoder = Encoder(
            vocab_size=src_vocab_size,
            d_model=d_model,
            num_layers=num_encoder_layers,
            num_heads=num_heads,
            d_ff=d_ff,
            max_seq_len=max_seq_length,
            dropout=dropout
        )
        
        self.decoder = Decoder(
            vocab_size=tgt_vocab_size,
            d_model=d_model,
            num_layers=num_decoder_layers,
            num_heads=num_heads,
            d_ff=d_ff,
            max_seq_len=max_seq_length,
            dropout=dropout
        )
        
        self.final_layer = nn.Linear(d_model, tgt_vocab_size)
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None,
                src_padding_mask=None, tgt_padding_mask=None,
                memory_key_padding_mask=None):
                
        # Process padding masks for attention
        if src_padding_mask is not None:
            src_mask = self.expand_mask(src_padding_mask)
        
        if tgt_padding_mask is not None:
            tgt_self_mask = self.expand_mask(tgt_padding_mask)
            if tgt_mask is not None:
                # Create a batched version of the causal mask
                batch_size = tgt_padding_mask.size(0)
                seq_len = tgt_padding_mask.size(-1)
                batched_causal_mask = tgt_mask[:seq_len, :seq_len].unsqueeze(0).unsqueeze(0).repeat(batch_size, 1, 1, 1)
                
                # Both masks have True where we should mask
                tgt_self_mask = tgt_self_mask | batched_causal_mask
        else:
            # If no padding mask, still need to expand causal mask for batch
            if tgt_mask is not None:
                batch_size = src.size(0)
                seq_len = tgt.size(1)
                tgt_self_mask = tgt_mask[:seq_len, :seq_len].unsqueeze(0).unsqueeze(0).repeat(batch_size, 1, 1, 1)
            else:
                tgt_self_mask = None
            
        # Encoder-decoder cross attention mask
        cross_mask = memory_key_padding_mask
        if cross_mask is not None:
            cross_mask = self.expand_mask(cross_mask)
        
        # Encode source sequence
        enc_output = self.encoder(src, src_mask)
        
        # Decode target sequence
        dec_output = self.decoder(tgt, enc_output, tgt_self_mask, cross_mask)
        
        # Final linear layer
        output = self.final_layer(dec_output)
        
        return output
    
    def expand_mask(self, mask):
        # Convert padding mask [batch_size, seq_len] to attention mask [batch_size, 1, 1, seq_len]
        return mask.unsqueeze(1).unsqueeze(2)
    
    def convert_mask_shape(self, tgt_mask, padding_mask):
        # For a causal mask, we need to reform it to match the padding mask
        batch_size = padding_mask.size(0)
        seq_len = padding_mask.size(-1)
        
        # Reshape causal mask to right dimensions
        causal = tgt_mask[:seq_len, :seq_len]  # Ensure correct sequence length
        
        # Make batch dimension the first dimension (for broadcasting)
        # [seq_len, seq_len] -> [1, 1, seq_len, seq_len] -> [batch_size, 1, 1, seq_len]
        return causal.unsqueeze(0).unsqueeze(0).repeat(batch_size, 1, 1, 1)
    
    def generate_square_subsequent_mask(self, sz):
        # Generate causal mask
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask.bool()  # Return as boolean mask
    
    def create_padding_mask(self, seq, pad_idx):
        # Create padding mask - True for padding positions (to be masked out)
        return (seq == pad_idx)  # [batch_size, seq_len]

# Init model

## Using Library

In [ ]:
# Initialize model
transformer = TransformerModel(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=EMBEDDING_SIZE,
    nhead=NUM_HEADS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=FFN_HID_DIM,
    max_seq_length=MAX_SEQ_LEN,
    dropout=0.1
).to(device)

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(transformer.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-9)

# Helper function for creating masks
def create_masks(src, tgt):
    src_padding_mask = transformer.create_padding_mask(src, PAD_IDX)
    tgt_padding_mask = transformer.create_padding_mask(tgt, PAD_IDX)
    
    tgt_len = tgt.shape[1]
    tgt_mask = transformer.generate_square_subsequent_mask(tgt_len).to(device)
    
    return src_padding_mask, tgt_padding_mask, tgt_mask

## Scrath Implementation

In [ ]:
# Initialize model
transformer = CustomTransformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=EMBEDDING_SIZE,
    num_heads=NUM_HEADS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    d_ff=FFN_HID_DIM,
    max_seq_length=MAX_SEQ_LEN,
    dropout=0.1
).to(device)

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(transformer.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-9)

# Function to create masks properly
def create_masks(src, tgt_input):
    batch_size, src_len = src.size()
    batch_size, tgt_len = tgt_input.size()
    
    # 1. Source padding mask (1s where padded, 0s elsewhere)
    src_padding_mask = (src == PAD_IDX)  # [batch_size, src_len]
    
    # 2. Target padding mask (1s where padded, 0s elsewhere)
    tgt_padding_mask = (tgt_input == PAD_IDX)  # [batch_size, tgt_len]
    
    # 3. Target causal mask (prevents attending to future tokens)
    # Shape: [tgt_len, tgt_len] with 1s in upper triangle
    tgt_causal_mask = torch.triu(
        torch.ones(tgt_len, tgt_len, device=device), diagonal=1
    ).bool()
    
    return src_padding_mask, tgt_padding_mask, tgt_causal_mask 

# Training

## Loss before training

In [ ]:
# Evaluate initial loss before training
def evaluate(model, dataloader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in dataloader:
            src = src.to(device)
            tgt = tgt.to(device)
            
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]
            
            # Get masks
            src_padding_mask, tgt_padding_mask, tgt_causal_mask = create_masks(src, tgt_input)
            
            # Forward pass with appropriate masks
            output = model(
                src=src,
                tgt=tgt_input,
                src_mask=None,  # No need for special source mask beyond padding
                tgt_mask=tgt_causal_mask,  # Causal mask for decoder self-attention
                src_padding_mask=src_padding_mask,  # Padding mask for encoder
                tgt_padding_mask=tgt_padding_mask,  # Padding mask for decoder
                memory_key_padding_mask=src_padding_mask  # Source padding mask for cross-attention
            )
            
            loss = criterion(output.reshape(-1, tgt_vocab_size), tgt_output.reshape(-1))
            total_loss += loss.item()
    
    return total_loss / len(dataloader)

# Evaluate initial loss
print("Evaluating initial loss before training...")
initial_val_loss = evaluate(transformer, val_loader)
print(f"Initial validation loss: {initial_val_loss:.4f}")

## Training

In [ ]:
# Training loop
transformer.train()
best_val_loss = float('inf')

print("Starting training...")
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    transformer.train()
    
    for batch_idx, (src, tgt) in enumerate(train_loader):
        src = src.to(device)
        tgt = tgt.to(device)
        
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        
        # Create masks
        src_padding_mask, tgt_padding_mask, tgt_mask = create_masks(src, tgt_input)
        
        # Forward pass
        optimizer.zero_grad()
        output = transformer(
            src, tgt_input, 
            tgt_mask=tgt_mask,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )
        
        # Compute loss
        loss = criterion(output.reshape(-1, tgt_vocab_size), tgt_output.reshape(-1))
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f"Epoch: {epoch+1}/{NUM_EPOCHS}, Batch: {batch_idx}/{len(train_loader)}, " 
                  f"Loss: {loss.item():.4f}")
    
    # Validation
    val_loss = evaluate(transformer, val_loader)
    print(f"Epoch: {epoch+1}/{NUM_EPOCHS}, Training Loss: {epoch_loss/len(train_loader):.4f}, "
          f"Validation Loss: {val_loss:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(transformer.state_dict(), 'models/transformer_best.pth')
        print(f"Saved new best model with validation loss: {val_loss:.4f}")

print("Training complete!")

## Load best model

In [ ]:
# Load best model and evaluate on test set
transformer.load_state_dict(torch.load('models/transformer_best.pth'))
test_loss = evaluate(transformer, test_loader)
print(f"Test loss: {test_loss:.4f}")